# Driver Drowsiness Detection Using Deep Learning Techniques
**Week 1 — Environment Setup & Exploratory Data Analysis**  
NTCC Project | Amity School of Engineering & Technology | May 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import zipfile
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf

print(f'TensorFlow : {tf.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'NumPy      : {np.__version__}')

In [ ]:
from google.colab import files
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
DATA_DIR   = f'{DRIVE_PATH}/data'
os.makedirs(f'{DRIVE_PATH}/results', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)

if not os.path.exists(DATA_DIR) or len(os.listdir(DATA_DIR)) == 0:
    os.system('kaggle datasets download -d dheerajperumandla/drowsiness-dataset')
    with zipfile.ZipFile('drowsiness-dataset.zip', 'r') as z:
        z.extractall('/content/dataset')
    import shutil
    shutil.copytree('/content/dataset', DATA_DIR, dirs_exist_ok=True)
    print('Dataset downloaded and saved to Drive')
else:
    print('Loaded from Drive')

print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
classes = [d for d in os.listdir(DATA_DIR)
           if os.path.isdir(os.path.join(DATA_DIR, d))]

class_counts = {}
for cls in classes:
    imgs = [f for f in os.listdir(os.path.join(DATA_DIR, cls))
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[cls] = len(imgs)
    print(f'{cls:20s}: {len(imgs)} images')

print(f'\nTotal : {sum(class_counts.values())} images')
print(f'Classes: {len(classes)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E53935', '#1E88E5', '#FF8F00', '#43A047']
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=colors[:len(class_counts)], edgecolor='white')
for bar, val in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 8, str(val), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Class Distribution', fontsize=13)
ax.set_ylabel('Number of Images')
ax.set_ylim(0, max(class_counts.values()) * 1.2)
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results/class_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(classes), 5, figsize=(13, 3 * len(classes)))
fig.suptitle('Sample Images Per Class', fontsize=13)
for row, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
    for col, img_name in enumerate(imgs):
        img = plt.imread(os.path.join(cls_path, img_name))
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 2:
            axes[row][col].set_title(cls, fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results/sample_images.png', dpi=150)
plt.show()

In [ ]:
size_data = []
for cls in classes:
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:50]
    for img_name in imgs:
        img = cv2.imread(os.path.join(cls_path, img_name))
        if img is not None:
            h, w = img.shape[:2]
            size_data.append({'class': cls, 'height': h, 'width': w})

df_sizes = pd.DataFrame(size_data)
print(df_sizes.groupby('class')[['height','width']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(14, 4))
colors = ['#E53935', '#1E88E5', '#FF8F00', '#43A047']
for idx, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:100]
    brightness = []
    for img_name in imgs:
        img = cv2.imread(os.path.join(cls_path, img_name), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            brightness.append(np.mean(img))
    axes[idx].hist(brightness, bins=20, color=colors[idx], alpha=0.85, edgecolor='white')
    axes[idx].axvline(np.mean(brightness), color='black', linestyle='--', lw=1.5)
    axes[idx].set_title(f'{cls}\nMean: {np.mean(brightness):.1f}', fontsize=10)
    axes[idx].set_xlabel('Mean pixel value')
fig.suptitle('Pixel Brightness Distribution Per Class', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results/brightness_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(classes), 8, figsize=(16, 4 * len(classes)))
fig.suptitle('Preprocessed images — 64x64 grayscale, normalised', fontsize=12)
for row, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:8]
    for col, img_name in enumerate(imgs):
        img = cv2.imread(os.path.join(cls_path, img_name), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (64, 64)) / 255.0
            axes[row][col].imshow(img, cmap='gray', vmin=0, vmax=1)
            axes[row][col].axis('off')
            if col == 3:
                axes[row][col].set_title(cls, fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results/preprocessed_preview.png', dpi=150)
plt.show()

In [ ]:
print('Dataset      :', 'dheerajperumandla/drowsiness-dataset (Kaggle)')
print('Total images :', sum(class_counts.values()))
print('Classes      :', list(class_counts.keys()))
print('Balance      :', 'Balanced' if max(class_counts.values()) - min(class_counts.values()) < 50 else 'Imbalanced')
print('Planned input: 64x64 grayscale, normalised 0-1')
print('Split        : 70% train / 15% val / 15% test (Week 2)')

In [ ]:
papers = [
    {'Title':'Real-Time Drowsiness Detection Using EAR and Facial Landmark Detection',
     'Authors':'Prerana et al.','Year':2024,'Method':'dlib + EAR threshold',
     'Accuracy':'91%','Limitation':'Fixed threshold, no CNN, fails with glasses'},
    {'Title':'CNN + MAR-Based Embedded Drowsiness Detection',
     'Authors':'Espinosa et al.','Year':2024,'Method':'CNN + EAR/MAR on Jetson Nano',
     'Accuracy':'97.44%','Limitation':'Specialised hardware, no web interface'},
    {'Title':'Personalised EAR/MAR Thresholds + CNN Classification',
     'Authors':'Sanchez-Gendriz et al.','Year':2025,'Method':'Personalised EAR/MAR + CNN',
     'Accuracy':'94%','Limitation':'No head pose, no app interface'},
    {'Title':'Driver Monitoring Using MediaPipe + MobileNetV2',
     'Authors':'Rosero-Montalvo et al.','Year':2026,'Method':'MediaPipe + MobileNetV2 + EAR+MAR+HeadPose',
     'Accuracy':'88.89%','Limitation':'No session logging, no deployment'},
    {'Title':'CNN Eye State Classification with MediaPipe',
     'Authors':'Castro-Ospina et al.','Year':2023,'Method':'ResNet50V2 + VGG16 + InceptionV3',
     'Accuracy':'99.71%','Limitation':'Too heavy for real-time on regular hardware'},
    {'Title':'Real-Time Transformer-Based Drowsiness Detection',
     'Authors':'Jarndal et al.','Year':2025,'Method':'ViT + Swin Transformer',
     'Accuracy':'99.15%','Limitation':'Too heavy for real-time PC deployment'},
]
df_lit = pd.DataFrame(papers)
df_lit.to_csv(f'{DRIVE_PATH}/results/literature_review.csv', index=False)
df_lit

In [ ]:
import shutil
from datetime import datetime

REPO_PATH = '/content/Driver-Drowsiness-Detection-Using-Deep-Learning-Techniques'
NOTEBOOK  = 'Week1_EDA.ipynb'

shutil.copy(f'/content/{NOTEBOOK}', f'{REPO_PATH}/notebooks/{NOTEBOOK}')
os.chdir(REPO_PATH)
os.system('git add .')
os.system(f'git commit -m "Week 1: EDA — {datetime.now().strftime("%d %b %Y")}'+'"')
print(os.popen('git push 2>&1').read())